# Case Técnico — Analista de Dados (Produto & Growth)
## the news — Dataset Palavritas

**Pergunta do Head de Produto:** O que está determinando se um usuário volta a jogar — e o que podemos fazer para aumentar isso?

Este notebook está organizado em três partes, seguindo a estrutura do case:

1. **Entrega 1** — Limpeza e diagnóstico dos dados
2. **Entrega 2** — Análise exploratória e estatística
3. **Entrega 3** — Proposta de ação


## Setup

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

pd.set_option('display.float_format', lambda x: f'{x:.3f}')


In [2]:
sessions = pd.read_csv('palavritas_sessions.csv')
attempts = pd.read_csv('palavritas_attempts.csv')
profile  = pd.read_csv('user_profile.csv')

print('sessions:', sessions.shape)
print('attempts:', attempts.shape)
print('profile:', profile.shape)


sessions: (41157, 13)
attempts: (147270, 5)
profile: (800, 15)


---
# Entrega 1 — Limpeza e Diagnóstico

Antes de qualquer análise, mapeei a estrutura dos três arquivos e investiguei problemas de qualidade que poderiam distorcer os resultados se não fossem tratados.


### 1.1 Estrutura geral e nulos

In [3]:
print('=== Tipos de dado (sessions) ===')
print(sessions.dtypes)
print()
print('=== Tipos de dado (profile) ===')
print(profile.dtypes)


=== Tipos de dado (sessions) ===
session_id                       str
user_id                          str
word                             str
word_date                        str
attempts                       int64
result                           str
time_to_complete_sec           int64
device                           str
session_hour                   int64
streak_day                     int64
played_next_day                 bool
newsletter_open_before_game     bool
active_d30                      bool
dtype: object

=== Tipos de dado (profile) ===
user_id                      str
age_range                    str
state                        str
city                         str
salary_range                 str
job_role                     str
sector                       str
company_size                 str
orders_food_delivery         str
food_delivery_freq_week    int64
food_delivery_platform       str
primary_device               str
plays_other_word_games      bool
typical_pl

In [4]:
print('=== Nulos por coluna — sessions ===')
print(sessions.isnull().sum())
print()
print('=== Nulos por coluna — profile ===')
print(profile.isnull().sum())


=== Nulos por coluna — sessions ===
session_id                      0
user_id                         0
word                            0
word_date                       0
attempts                        0
result                         63
time_to_complete_sec            0
device                          0
session_hour                    0
streak_day                      0
played_next_day                 0
newsletter_open_before_game     0
active_d30                      0
dtype: int64

=== Nulos por coluna — profile ===
user_id                      0
age_range                  117
state                        0
city                       297
salary_range               193
job_role                     0
sector                       0
company_size                 0
orders_food_delivery         0
food_delivery_freq_week      0
food_delivery_platform       0
primary_device               0
plays_other_word_games       0
typical_play_time            0
newsletter_subscriber        0
dtype: i

**Achado:** `result` tem 63 nulos em `sessions`. Em `profile`, `age_range` (117), `city` (297) e `salary_range` (193) têm nulos relevantes — optei por tratá-los como categoria "não informado" nas análises em vez de descartar as linhas, já que excluir reduziria a base de perfil em até ~25%.

### 1.2 Duplicatas

In [5]:
dup_ids = sessions[sessions['session_id'].duplicated(keep=False)]
print('Linhas envolvidas em duplicatas de session_id:', len(dup_ids))
print('Linhas 100% idênticas (todas as colunas):', sessions.duplicated().sum())


Linhas envolvidas em duplicatas de session_id: 2396
Linhas 100% idênticas (todas as colunas): 1198


**Achado:** 1.198 linhas estão 100% duplicadas (mesmo `session_id`, todas as colunas idênticas) — indicando duplicação na geração/exportação dos dados, não eventos de jogo distintos. **Decisão:** remover as duplicatas exatas.

### 1.3 Formato de data inconsistente

In [6]:
import re

def date_format(s):
    if re.match(r'^\d{4}-\d{2}-\d{2}$', s):
        return 'YYYY-MM-DD'
    elif re.match(r'^\d{2}/\d{2}/\d{4}$', s):
        return 'DD/MM/YYYY'
    return 'outro'

print(sessions['word_date'].apply(date_format).value_counts())


word_date
YYYY-MM-DD    34995
DD/MM/YYYY     6162
Name: count, dtype: int64


**Achado:** a coluna `word_date` mistura dois formatos — `YYYY-MM-DD` (maioria) e `DD/MM/YYYY` (~15% das linhas). Sem corrigir isso, qualquer análise temporal ficaria errada (datas mal interpretadas). **Decisão:** fazer o parsing com formato misto (`format='mixed', dayfirst=True`).

### 1.4 Valores impossíveis em `attempts`

In [7]:
print(pd.crosstab(sessions['attempts'], sessions['result'], dropna=False))


result     lose   win  NaN
attempts                  
0             8     8   43
1             0  8331    6
2             0  8273    3
3             0  8169    6
6         16271     0    5
7             6     6    0
8             8    14    0


**Achado crítico:** não existe nenhum registro com `attempts = 4` ou `attempts = 5` em todo o dataset. Toda vitória (`win`) ocorre apenas com 1, 2 ou 3 tentativas; toda derrota (`lose`) está marcada com `attempts = 6`, independentemente de quantas tentativas reais o usuário usou. Isso indica que a coluna `attempts` não registra fielmente o número de tentativas usadas — parece usar "6" como valor padrão para derrota, em vez da contagem real. Além disso, há 59 sessões com `attempts = 0` e 34 com `attempts = 7` ou `8`, valores fora do intervalo possível do jogo (1–6).

**Decisão:** tratar `attempts` fora do intervalo 1–6 como inválido (`NaN`), e usar essa coluna com cautela em qualquer análise que dependa de granularidade fina de tentativas — ela não é confiável para diferenciar, por exemplo, "venceu na 4ª tentativa" de "venceu na 6ª".

### 1.5 Outras inconsistências de captura

In [8]:
print('Sessões com tempo negativo (time_to_complete_sec < 0):', (sessions['time_to_complete_sec'] < 0).sum())
print()
print('Variações de capitalização em device:')
print(sessions['device'].value_counts())


Sessões com tempo negativo (time_to_complete_sec < 0): 30

Variações de capitalização em device:
device
iOS        20022
Android    16687
ios         1691
android     1482
IOS          713
ANDROID      562
Name: count, dtype: int64


**Achados:** 30 sessões têm `time_to_complete_sec` negativo (fisicamente impossível) — tratadas como inválidas (`NaN`). A coluna `device` tem 6 variações de texto para apenas 2 dispositivos reais (`iOS`/`ios`/`IOS`, `Android`/`android`/`ANDROID`) — normalizada para 2 categorias.

### 1.6 Integridade referencial entre `sessions` e `attempts`

In [9]:
sessions_ids = set(sessions['session_id'])
attempts_ids = set(attempts['session_id'])

print('session_ids em attempts sem correspondência em sessions:', len(attempts_ids - sessions_ids))
print('session_ids em sessions sem nenhuma linha em attempts:', len(sessions_ids - attempts_ids))

attempt_counts = attempts.groupby('session_id').size().rename('attempts_real')
check = sessions[['session_id','attempts']].drop_duplicates('session_id').merge(attempt_counts, on='session_id', how='left')
check['diff'] = check['attempts'] - check['attempts_real']
print()
print('Divergência entre attempts (sessions) e contagem real (attempts):')
print(check['diff'].value_counts(dropna=False).sort_index())


session_ids em attempts sem correspondência em sessions: 200
session_ids em sessions sem nenhuma linha em attempts: 90



Divergência entre attempts (sessions) e contagem real (attempts):
diff
-6.000      465
-3.000      225
-2.000      233
-1.000      272
0.000     38674
NaN          90
Name: count, dtype: int64


**Achado:** 200 `session_id` em `attempts` não existem em `sessions` (registros órfãos), e 90 sessões não têm nenhuma linha correspondente em `attempts`. Mesmo nas sessões que casam, em 1.195 casos o número declarado em `sessions.attempts` não bate com a contagem real de tentativas — reforçando o achado do item 1.4: essa coluna não é confiável para análises de granularidade fina. **Decisão:** para a Entrega 2, usar a tabela `attempts` apenas como apoio exploratório (ex.: dificuldade da palavra), e não como fonte de verdade para "número de tentativas".

### 1.7 Inconsistências de categoria em `user_profile`

In [10]:
print('job_role - valores únicos antes da normalização:', profile['job_role'].nunique())
print(sorted(profile['job_role'].unique())[:10], '...')
print()
print('orders_food_delivery - valores únicos:')
print(profile['orders_food_delivery'].value_counts(dropna=False))


job_role - valores únicos antes da normalização: 40
['Analista', 'Analista Sênior', 'Analista de Dados', 'Consultor', 'Consultor Sênior', 'Consultor de Dados', 'Coordenador', 'Coordenador Sênior', 'Coordenador de Dados', 'Desenvolvedor'] ...

orders_food_delivery - valores únicos:
orders_food_delivery
True     658
False    108
sim       27
não        7
Name: count, dtype: int64


**Achados:** `job_role` tinha 40 valores únicos por conta de inconsistência de capitalização (ex.: "Gerente" e "gerente" contados separadamente) — após normalizar para Title Case, restaram 30 categorias genuinamente distintas (10 cargos-base × 3 variações reais: padrão / "De Dados" / "Sênior"). `orders_food_delivery` mistura tipos: booleano (`True`/`False`) e texto em português (`"sim"`/`"não"`) para a mesma variável binária — unificado em booleano único.

**Achado de cobertura:** apenas 800 dos 1.200 usuários únicos em `sessions` (66,7%) têm dados de perfil. Qualquer cruzamento com perfil demográfico/comportamental cobre só 2/3 da base, e pode haver viés (quem respondeu à pesquisa pode não representar o usuário médio).

### 1.8 Limpeza aplicada

In [11]:
# Sessions
sessions_clean = sessions.drop_duplicates().copy()
sessions_clean['word_date'] = pd.to_datetime(sessions_clean['word_date'], format='mixed', dayfirst=True)
sessions_clean['device'] = sessions_clean['device'].str.lower().map({'ios': 'iOS', 'android': 'Android'})
sessions_clean['attempts_valid'] = sessions_clean['attempts'].between(1, 6)
sessions_clean.loc[~sessions_clean['attempts_valid'], 'attempts'] = np.nan
sessions_clean.loc[sessions_clean['time_to_complete_sec'] < 0, 'time_to_complete_sec'] = np.nan

# Profile
profile_clean = profile.copy()
text_cols = ['job_role', 'sector', 'company_size', 'salary_range', 'age_range',
             'food_delivery_platform', 'primary_device', 'typical_play_time', 'state', 'city']
for col in text_cols:
    profile_clean[col] = profile_clean[col].astype(str).str.strip()
    profile_clean.loc[profile_clean[col].isin(['nan', 'None', '']), col] = np.nan
profile_clean['job_role'] = profile_clean['job_role'].str.title()

map_bool = {'True': True, 'False': False, 'sim': True, 'não': False}
profile_clean['orders_food_delivery'] = profile_clean['orders_food_delivery'].astype(str).map(map_bool)

print('Shape sessions pós-limpeza:', sessions_clean.shape, '(', sessions.shape[0] - sessions_clean.shape[0], 'linhas removidas por duplicidade )')
print('job_role - valores únicos pós-normalização:', profile_clean['job_role'].nunique())


Shape sessions pós-limpeza: (39959, 14) ( 1198 linhas removidas por duplicidade )
job_role - valores únicos pós-normalização: 30


---
# Entrega 2 — Análise

**Pergunta:** quais variáveis mais se correlacionam com o usuário estar ativo em D30 (`active_d30`) e voltar a jogar no dia seguinte (`played_next_day`)?

Explorei: horário de jogo, palavra do dia, perfil do usuário (setor, salário, device), frequência de food delivery, e relação com a newsletter.


### 2.1 Taxas gerais (baseline) e join com perfil

In [12]:
print('Taxa geral active_d30:', round(sessions_clean['active_d30'].mean(), 3))
print('Taxa geral played_next_day:', round(sessions_clean['played_next_day'].mean(), 3))

df = sessions_clean.merge(profile_clean, on='user_id', how='left')
print('Sessões com perfil disponível:', df['sector'].notna().sum(), 'de', len(df))


Taxa geral active_d30: 0.319
Taxa geral played_next_day: 0.221
Sessões com perfil disponível: 26759 de 39959


### 2.2 Horário de jogo

In [13]:
g = df.groupby('session_hour')[['active_d30','played_next_day']].mean().round(3)
g['n'] = df.groupby('session_hour').size()
print(g)


              active_d30  played_next_day     n
session_hour                                   
6                  0.364            0.212  4161
7                  0.385            0.229  4263
8                  0.379            0.219  4171
12                 0.275            0.214  1391
13                 0.291            0.228  1404
14                 0.280            0.239  1391
15                 0.342            0.247  1414
16                 0.287            0.199  1354
17                 0.310            0.226  1375
18                 0.275            0.212  2658
19                 0.297            0.207  2661
20                 0.280            0.219  2585
21                 0.297            0.225  2787
22                 0.289            0.217  4167
23                 0.305            0.236  4177


**Achado:** sessões realizadas entre 6h-8h da manhã têm `active_d30` claramente mais alto (0,36–0,39) que o restante do dia (0,27–0,31). Vale notar que não há registros nas horas 9h-11h e 0h-5h — um padrão estrutural do dataset (provavelmente 3 janelas de uso: manhã/tarde/noite), não um problema de qualidade.

### 2.3 Newsletter, resultado e sequência (streak)

In [14]:
for col in ['newsletter_open_before_game', 'result', 'streak_day']:
    print(f'=== active_d30 / played_next_day por {col} ===')
    g = df.groupby(col)[['active_d30','played_next_day']].mean().round(3)
    g['n'] = df.groupby(col).size()
    print(g)
    print()


=== active_d30 / played_next_day por newsletter_open_before_game ===
                             active_d30  played_next_day      n
newsletter_open_before_game                                    
False                             0.305            0.223  32280
True                              0.378            0.215   7679

=== active_d30 / played_next_day por result ===
        active_d30  played_next_day      n
result                                    
lose         0.308            0.225  15828
win          0.326            0.219  24071

=== active_d30 / played_next_day por streak_day ===
            active_d30  played_next_day      n
streak_day                                    
1                0.319            0.217  31110
2                0.319            0.235   6743
3                0.315            0.238   1583
4                0.340            0.277    376
5                0.356            0.298    104
6                0.355            0.290     31
7                0.333   

**Achados:**
- `newsletter_open_before_game = True` → `active_d30` de 37,8% vs 30,5% para quem não abriu — a diferença mais expressiva encontrada até aqui.
- `result` (venceu/perdeu o dia) tem efeito mínimo sobre retenção.
- `streak_day` parece subir junto com `active_d30` na tabela simples, mas a maior parte da base está concentrada em `streak_day = 1` (31 mil de 40 mil linhas) — o efeito visual em sequências altas pode ser ruído de amostra pequena. Testei isso formalmente mais abaixo.

### 2.4 Verificação de confundimento — newsletter x horário matinal

In [15]:
ct = pd.crosstab(df['newsletter_open_before_game'], df['session_hour'].between(6, 8), normalize='index')
print('Proporção de sessões em horário matinal (6h-8h), por newsletter_open_before_game:')
print(ct.round(3))


Proporção de sessões em horário matinal (6h-8h), por newsletter_open_before_game:
session_hour                 False  True 
newsletter_open_before_game              
False                        0.848  0.152
True                         0.000  1.000


**Achado importante:** sempre que `newsletter_open_before_game = True`, a sessão ocorre necessariamente entre 6h-8h (100% dos casos). Ou seja, "abrir a newsletter antes de jogar" e "jogar de manhã" **não são dois sinais independentes — são a mesma população/comportamento**. Trato isso como um único achado (rotina matinal de newsletter + jogo), não dois achados separados.

### 2.5 Perfil do usuário (setor, salário, device e demais variáveis)

In [16]:
for col in ['sector','salary_range','company_size','typical_play_time','primary_device','orders_food_delivery']:
    g = df.groupby(col, dropna=False)[['active_d30','played_next_day']].mean().round(3)
    g['n'] = df.groupby(col, dropna=False).size()
    print(f'=== {col} ===')
    print(g.sort_values('active_d30', ascending=False))
    print()

print('Correlação food_delivery_freq_week x active_d30:', round(df[['food_delivery_freq_week','active_d30']].corr().iloc[0,1], 4))


=== sector ===
           active_d30  played_next_day      n
sector                                       
educação        0.338            0.236   3390
varejo          0.331            0.229   2414
marketing       0.324            0.212   2035
tech            0.323            0.233   6112
finanças        0.320            0.216   3779
direito         0.317            0.209   1401
outros          0.315            0.218   5075
NaN             0.313            0.218  13200
saúde           0.310            0.218   2553

=== salary_range ===
                active_d30  played_next_day      n
salary_range                                      
até R$2k             0.330            0.229   3320
R$2k-R$4k            0.327            0.232   5096
R$4k-R$6k            0.323            0.219   5402
NaN                  0.318            0.218  19617
R$6k-R$10k           0.312            0.216   3685
acima de R$10k       0.305            0.227   2839



=== company_size ===
               active_d30  played_next_day      n
company_size                                     
média               0.330            0.223   5825
MEI/micro           0.322            0.233   6017
pequena             0.320            0.217   4621
multinacional       0.320            0.220   4876
grande              0.319            0.221   5420
NaN                 0.313            0.218  13200

=== typical_play_time ===
                   active_d30  played_next_day      n
typical_play_time                                    
morning                 0.378            0.228   8558
NaN                     0.313            0.218  13200
afternoon               0.304            0.232   5719
night                   0.295            0.218   5247
evening                 0.290            0.215   7235

=== primary_device ===
                active_d30  played_next_day      n
primary_device                                    
iOS                  0.323            0.222  149

**Achados:** `typical_play_time = morning` confirma o padrão do item 2.4 (active_d30 de 37,8% vs 29-30% nos demais períodos). As demais variáveis de perfil — setor, salário, porte de empresa, device, pedir delivery — mostram diferenças pequenas e sem padrão claro de negócio. Frequência de food delivery não correlaciona com retenção (r ≈ 0,004).

### 2.6 Dificuldade da palavra do dia (exploratório)

In [17]:
word_level = df.groupby('word').agg(
    win_rate=('result', lambda x: (x == 'win').mean()),
    avg_attempts=('attempts', 'mean'),
    avg_time=('time_to_complete_sec', 'mean'),
    active_d30=('active_d30', 'mean'),
    n=('active_d30', 'size')
).query('n >= 30')

print('Número de palavras únicas analisadas:', len(word_level))
print()
print('Correlação (nível palavra) com active_d30:')
print(word_level[['win_rate','avg_attempts','avg_time']].corrwith(word_level['active_d30']).round(3))


Número de palavras únicas analisadas: 30

Correlação (nível palavra) com active_d30:
win_rate       -0.258
avg_attempts    0.258
avg_time       -0.226
dtype: float64


**Achado exploratório (baixa confiança estatística):** com apenas 30 palavras únicas, qualquer correlação no nível de palavra tem pouco poder estatístico. Ainda assim, o padrão observado é interessante: palavras que exigem mais tentativas (`avg_attempts`, r ≈ +0,26) e têm menor taxa de vitória (`win_rate`, r ≈ -0,26) associam-se a retenção levemente maior — sugerindo que algum nível de desafio pode reforçar engajamento. Já tempo médio de conclusão mais longo (`avg_time`, r ≈ -0,23) associa-se a retenção menor, possivelmente capturando frustração em vez de desafio saudável. **Não uso este achado como base da proposta**, por ser exploratório — fica como hipótese para investigação futura.

### 2.7 Testes de significância estatística

In [18]:
# Chi-square: newsletter_open_before_game x active_d30
ct1 = pd.crosstab(df['newsletter_open_before_game'], df['active_d30'])
chi2, p, dof, exp = stats.chi2_contingency(ct1)
print(f'Chi-square newsletter_open_before_game x active_d30: chi2={chi2:.1f}, p={p:.2e}')

# Correlação streak_day x active_d30
corr, pcorr = stats.pointbiserialr(df['active_d30'], df['streak_day'])
print(f'Correlação streak_day x active_d30: r={corr:.3f}, p={pcorr:.3f}  -> não significativo')


Chi-square newsletter_open_before_game x active_d30: chi2=151.4, p=8.67e-35
Correlação streak_day x active_d30: r=0.002, p=0.754  -> não significativo


### 2.8 Regressão logística — isolando o efeito controlando confundidores

In [19]:
model_df = df[['active_d30','newsletter_open_before_game','result','device','streak_day']].dropna().copy()
model_df['active_d30'] = model_df['active_d30'].astype(int)
model_df['newsletter_open_before_game'] = model_df['newsletter_open_before_game'].astype(int)
model_df['result_win'] = (model_df['result'] == 'win').astype(int)
model_df['device_ios'] = (model_df['device'] == 'iOS').astype(int)

logit = smf.logit(
    'active_d30 ~ newsletter_open_before_game + result_win + device_ios + streak_day',
    data=model_df
).fit(disp=0)

print(logit.summary())
print()
print('Odds Ratios:')
print(np.exp(logit.params).round(3))


                           Logit Regression Results                           
Dep. Variable:             active_d30   No. Observations:                39899
Model:                          Logit   Df Residuals:                    39894
Method:                           MLE   Df Model:                            4
Date:                Sun, 28 Jun 2026   Pseudo R-squ.:                0.003029
Time:                        20:37:54   Log-Likelihood:                -24915.
converged:                       True   LL-Null:                       -24991.
Covariance Type:            nonrobust   LLR p-value:                 1.033e-31
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                      -0.8591      0.031    -27.876      0.000      -0.919      -0.799
newsletter_open_before_game     0.3165      0.027     11.779      0.000       0.

**Conclusão da Entrega 2:** controlando resultado do dia, device e sequência, o único preditor que se mantém estatisticamente significativo é `newsletter_open_before_game` (p < 0,001), com odds ratio de 1,37 — ou seja, abrir a newsletter antes de jogar está associado a 37% mais chance de estar ativo em D30. `streak_day`, `result` e `device` não são significativos no modelo controlado (apesar de `streak_day` parecer relevante na tabela simples da seção 2.3 — isso era ruído).

**Ressalva importante:** esse é um achado de correlação, não de causalidade. É possível que esse grupo seja simplesmente mais engajado com o produto como um todo (e por isso abre a newsletter e joga), e não que a newsletter "cause" a retenção. Essa distinção orienta a proposta a seguir.

---
# Entrega 3 — Proposta

### Hipótese
A correlação observada entre abrir a newsletter antes de jogar e a retenção em D30 pode refletir um perfil de usuário já mais engajado com o produto como um todo — não necessariamente um efeito causal da newsletter sobre a retenção. Usuários que consomem a newsletter pela manhã, antes do jogo, podem já estar numa rotina diária mais forte de consumo do the news, e o Palavritas seria apenas mais uma expressão desse hábito, não a causa dele.

### Ação
Antes de investir em mudanças de produto (ex.: CTA da newsletter para o jogo, notificações push), recomendo rodar um teste controlado pequeno e barato para isolar causalidade: selecionar aleatoriamente um grupo de usuários que normalmente **não** abrem a newsletter antes de jogar, e enviar um nudge (notificação push ou e-mail) incentivando-os a abrir a newsletter antes de jogar por um período definido (ex.: 2 semanas). Comparar esse grupo a um grupo controle equivalente que não recebe o nudge.

### Critério de sucesso
Se o grupo que recebeu o nudge apresentar `active_d30` significativamente maior que o grupo controle (testado via chi-quadrado ou regressão logística, como feito na Entrega 2), isso confirma um efeito causal e justifica investimento maior em mudanças de produto que incentivem a rotina newsletter → jogo. Se não houver diferença significativa, o achado da Entrega 2 é melhor interpretado como um marcador de engajamento pré-existente, e o esforço de produto deve ser redirecionado para outras alavancas de retenção.
